# Batch routing (GPU / cost constrained)

Run **top to bottom** after `cd` to `llm_routing/notebooks` (or `llm_routing/`).

1. **Setup** — paths, imports, `DATA_DIR`, `RESULTS_DIR`  
2. **Config** — `routers` (CSV paths under `results/performance_estimates/`)  
3. **Optimization** — writes `optimization_results_*.csv` into `results/routing_results/` (can take **several minutes**)  
4. **Baselines** — writes `baseline_results_*.csv` into the same directory  
5. **Plots** — loads those CSVs from `results/routing_results/`; writes PDFs there (skips if inputs are missing)


In [ ]:
import sys
from pathlib import Path

# --- locate llm_routing/ (contains routing/solver/experiments.py) ---
_here = Path.cwd().resolve()
LLM_ROUTING_DIR = None
for candidate in (_here, _here.parent, _here.parent.parent):
    if (candidate / "routing" / "solver" / "experiments.py").is_file():
        LLM_ROUTING_DIR = candidate
        break
if LLM_ROUTING_DIR is None:
    raise RuntimeError(
        "Could not find llm_routing (routing/solver/experiments.py). "
        f"cd to llm_routing/ or llm_routing/notebooks/. cwd={_here}"
    )

ROUTING_DIR = LLM_ROUTING_DIR / "routing"
NOTEBOOK_DIR = LLM_ROUTING_DIR / "notebooks"
for p in (LLM_ROUTING_DIR, ROUTING_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
if NOTEBOOK_DIR.is_dir() and str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import numpy as np

from routing.batch_optimization import (
    PERF_EST_DIR,
    RESULTS_DIR,
    create_model_configuration,
)
from solver.eval import evaluate_single_llm_baseline
from solver.experiments import (
    run_router_optimization_experiments,
    get_baseline_results,
)

DATA_DIR = PERF_EST_DIR

print("Paths")
print(f"  DATA_DIR (perf estimates): {DATA_DIR}")
print(f"  RESULTS_DIR (batch CSVs):  {RESULTS_DIR}")



In [ ]:
# Batch routing experiment config (test1)

test_name = "test1"

routers = [
    ("mirt", "performance_prediction", test_name, str(DATA_DIR / f"mirt_bert_{test_name}.csv")),
    ("xgboost", "main_model_prediction", test_name, str(DATA_DIR / f"xgboost_bootstrap_100_bert_{test_name}.csv")),
    ("xgboost", "bootstrap_quantile_10", test_name, str(DATA_DIR / f"xgboost_bootstrap_100_bert_{test_name}.csv")),
    ("knn_40", "main_model_prediction", test_name, str(DATA_DIR / f"knn_40_bootstrap_100_bert_{test_name}.csv")),
    ("knn_40", "bootstrap_quantile_10", test_name, str(DATA_DIR / f"knn_40_bootstrap_100_bert_{test_name}.csv")),
]

max_gpu_per_model = 8
max_cost_per_query_list = [1.0e-6, 3.0e-6]



In [ ]:
for batch_size in [100]:
    gpu_counts = np.linspace(0, batch_size, 10, dtype=int).tolist()

    all_results = run_router_optimization_experiments(
        routers=routers,
        gpu_counts=gpu_counts,
        max_cost_per_query_list=max_cost_per_query_list,
        batch_size=batch_size,
        create_model_configuration=create_model_configuration,
        train_ratio=0.1,
        seed=42,
        verbose=True,
        output_dir=str(RESULTS_DIR),
    )



In [ ]:
routing_csv_path = str(DATA_DIR / f"knn_40_bootstrap_100_bert_{test_name}.csv")

for batch_size in [100]:
    gpu_counts = np.linspace(0, batch_size, 10, dtype=int).tolist()
    baseline_csv_path = str(RESULTS_DIR / f"baseline_results_{batch_size}_{test_name}.csv")

    get_baseline_results(
        routing_csv_path=routing_csv_path,
        gpu_counts=gpu_counts,
        batch_size=batch_size,
        baseline_csv_path=baseline_csv_path,
        create_model_configuration=create_model_configuration,
        evaluate_single_llm_baseline=evaluate_single_llm_baseline,
        train_ratio=0.1,
        seed=42,
        verbose=True,
    )



In [ ]:
import importlib

import pandas as pd

import plotting
importlib.reload(plotting)
from plotting import plot_csv_comparison_with_baselines, plot_csv_focused_comparison

BATCH_CSV_DIR = RESULTS_DIR

batch_size = 100
test_name = "test1"
gpu_counts = np.linspace(0, batch_size, 10, dtype=int).tolist()

baseline_csv_path = BATCH_CSV_DIR / f"baseline_results_{batch_size}_{test_name}.csv"
if baseline_csv_path.exists():
    baseline_df = pd.read_csv(baseline_csv_path)
else:
    baseline_df = None
    print(f"(No baseline CSV yet: {baseline_csv_path.name} — run baseline cell first)\n")

csv_files = [
    (f"optimization_results_{batch_size}_mirt_performance_prediction_{test_name}_concurrency.csv", "MIRT", "performance_prediction"),
    (f"optimization_results_{batch_size}_xgboost_bootstrap_quantile_10_{test_name}_concurrency.csv", "XGBoost Robust Q10", "bootstrap_quantile_10"),
    (f"optimization_results_{batch_size}_xgboost_main_model_prediction_{test_name}_concurrency.csv", "XGBoost", "main_model_prediction"),
    (f"optimization_results_{batch_size}_knn_40_main_model_prediction_{test_name}_concurrency.csv", "kNN 40", "main_model_prediction"),
    (f"optimization_results_{batch_size}_knn_40_bootstrap_quantile_10_{test_name}_concurrency.csv", "kNN 40 Robust Q10", "bootstrap_quantile_10"),
]

loaded_results = {}
max_perf_by_router = []
cost = 3e-6

print("Loading optimization CSVs (results/routing_results)...")
print("=" * 70)

for csv_file, label, pred_label in csv_files:
    candidate = BATCH_CSV_DIR / csv_file
    if not candidate.exists():
        print(f"Skipping {label}: not found ({candidate.name})")
        continue
    df = pd.read_csv(candidate)
    filtered_df = df[(df["cost_budget"] - cost).abs() < 1e-10].copy()
    loaded_results[label] = {"df": filtered_df, "max_cost": cost, "pred_label": pred_label}
    max_tp_plot = filtered_df["test_performance"].max(skipna=True) if "test_performance" in filtered_df.columns else float("nan")
    max_perf_by_router.append((label, max_tp_plot))
    print(f"\n{label}: {candidate.name}")
    print(f"  cost_budget slice: {cost:.2e}")
    if pd.notna(max_tp_plot):
        print(f"  max test_performance on slice: {max_tp_plot:.4f}")
    print(f"  rows (non-NaN perf): {filtered_df.dropna(subset=['test_performance']).shape[0]}")

print("\n" + "=" * 70)
for lab, m in max_perf_by_router:
    print(f"  {lab}: {m:.4f}" if pd.notna(m) else f"  {lab}: (n/a)")
print("=" * 70)

if not loaded_results:
    print("No optimization CSVs found under RESULTS_DIR; run the optimization cell first.")
else:
    baseline_performance_threshold = 0.65
    plot_dir = RESULTS_DIR
    plot_dir.mkdir(parents=True, exist_ok=True)

    fig_full, axes_full = plot_csv_comparison_with_baselines(
        loaded_results=loaded_results,
        baseline_df=None,
        output_filename=str(plot_dir / f"batch_full_{batch_size}_cost_{cost}_{test_name}.pdf"),
        gpu_counts=gpu_counts,
        baseline_performance_threshold=baseline_performance_threshold,
        test_name=test_name,
    )

    fig_focused, ax_focused = plot_csv_focused_comparison(
        loaded_results=loaded_results,
        baseline_df=None,
        output_filename=str(plot_dir / f"batch_focused_{batch_size}_cost_{cost}_{test_name}.pdf"),
        gpu_counts=gpu_counts,
        baseline_performance_threshold=baseline_performance_threshold,
        test_name=test_name,
    )

    print("Plots written under:", plot_dir)

